# Predicting Outcomes of Patients with Cirrhosis

## Introduction
In this project, the task is to predict the outcomes of patients with cirrhosis using a multi-class classification approach. The dataset includes information about patients' medical histories and current conditions, with the goal of predicting one of three possible outcomes: Status_C, Status_CL, or Status_D. The model's performance is evaluated using multi-class logarithmic loss, which takes into account the predicted probabilities for each of the three outcomes. The data can be downloaded from the [Kaggle competition page](https://www.kaggle.com/competitions/playground-series-s3e26/data).


In [2]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [4]:
train = pd.read_csv("train.csv")
test =  pd.read_csv("test.csv")

In [6]:
train.head()

,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
0,0,999,D-penicillamine,21532,M,N,N,N,N,2.3,316.0,3.35,172.0,1601.0,179.80,63.0,394.0,9.7,3.0,D
1,1,2574,Placebo,19237,F,N,N,N,N,0.9,364.0,3.54,63.0,1440.0,134.85,88.0,361.0,11.0,3.0,C
2,2,3428,Placebo,13727,F,N,Y,Y,Y,3.3,299.0,3.55,131.0,1029.0,119.35,50.0,199.0,11.7,4.0,D
3,3,2576,Placebo,18460,F,N,N,N,N,0.6,256.0,3.50,58.0,1653.0,71.30,96.0,269.0,10.7,3.0,C
4,4,788,Placebo,16658,F,N,Y,N,N,1.1,346.0,3.65,63.0,1181.0,125.55,96.0,298.0,10.6,4.0,C


In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7905 entries, 0 to 7904
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             7905 non-null   int64  
 1   N_Days         7905 non-null   int64  
 2   Drug           7905 non-null   object 
 3   Age            7905 non-null   int64  
 4   Sex            7905 non-null   object 
 5   Ascites        7905 non-null   object 
 6   Hepatomegaly   7905 non-null   object 
 7   Spiders        7905 non-null   object 
 8   Edema          7905 non-null   object 
 9   Bilirubin      7905 non-null   float64
 10  Cholesterol    7905 non-null   float64
 11  Albumin        7905 non-null   float64
 12  Copper         7905 non-null   float64
 13  Alk_Phos       7905 non-null   float64
 14  SGOT           7905 non-null   float64
 15  Tryglicerides  7905 non-null   float64
 16  Platelets      7905 non-null   float64
 17  Prothrombin    7905 non-null   float64
 18  Stage   

In [10]:
train.isnull().sum()

id               0
N_Days           0
Drug             0
Age              0
Sex              0
Ascites          0
Hepatomegaly     0
Spiders          0
Edema            0
Bilirubin        0
Cholesterol      0
Albumin          0
Copper           0
Alk_Phos         0
SGOT             0
Tryglicerides    0
Platelets        0
Prothrombin      0
Stage            0
Status           0
dtype: int64

In [12]:
test.isnull().sum()

id               0
N_Days           0
Drug             0
Age              0
Sex              0
Ascites          0
Hepatomegaly     0
Spiders          0
Edema            0
Bilirubin        0
Cholesterol      0
Albumin          0
Copper           0
Alk_Phos         0
SGOT             0
Tryglicerides    0
Platelets        0
Prothrombin      0
Stage            0
dtype: int64

In [16]:
# categorical 
categorical_features = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Stage']
train = pd.get_dummies(train, columns=categorical_features, drop_first=True)
test = pd.get_dummies(test, columns=categorical_features, drop_first=True)

label_encoder = LabelEncoder()
train['Status'] = label_encoder.fit_transform(train['Status'])

class_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))

print("Label to Number Mapping:")
for label, number in class_mapping.items():
    print(f"{label}: {number}")

Label to Number Mapping:
C: 0
CL: 1
D: 2


In [18]:
X = train.drop(columns=['id', 'Status'])
y = train['Status']


In [20]:
test_df = test.drop(columns='id')


In [24]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=37)

In [26]:
numeric_features = ['N_Days', 'Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']

# Apply StandardScaler to numeric features
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_val[numeric_features] = scaler.transform(X_val[numeric_features])

test_df[numeric_features] = scaler.transform(test_df[numeric_features])

# Define the model (XGBoost Classifier)
model = XGBClassifier(random_state=37)

# Train the model
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [28]:
y_pred = model.predict(X_val)


In [30]:
print(classification_report(y_val, y_pred))
print(model.score(X_train, y_train))

              precision    recall  f1-score   support

           0       0.86      0.91      0.88      1251
           1       0.54      0.17      0.26        75
           2       0.79      0.77      0.78       651

    accuracy                           0.83      1977
   macro avg       0.73      0.62      0.64      1977
weighted avg       0.82      0.83      0.82      1977

0.997638326585695


In [36]:
prob = model.predict_proba(test_df)
Status_C = prob[:,0]
Status_CL = prob[:,1]
Status_D = prob[:,2]

In [38]:
data = {
    'id': test['id'],
    'Status_C': Status_C,
    'Status_CL': Status_CL,
    'Status_D': Status_D
}

submission = pd.DataFrame(data)
submission.to_csv('submission.csv', index=False)
submission.head(10)

,id,Status_C,Status_CL,Status_D
0,7905,0.712667,0.006572,0.280761
1,7906,0.713325,0.114574,0.172101
2,7907,0.006584,0.000621,0.992795
3,7908,0.991954,0.000284,0.007762
4,7909,0.972575,0.003414,0.024011
5,7910,0.998913,0.000125,0.000961
6,7911,0.994968,0.000092,0.004940
7,7912,0.014574,0.009407,0.976019
8,7913,0.001535,0.000062,0.998403
9,7914,0.834957,0.002470,0.162573


## Conclusion

In this project, the **XGBClassifier** was used to predict the outcomes of patients with cirrhosis. The model performed well on the validation set, achieving an accuracy of **83%**. The detailed performance metrics showed strong precision and recall for the majority class (Status_C), with a precision of **0.86** and recall of **0.91**. However, the model struggled with the minority class (Status_CL), where recall was lower at **0.17**, leading to a lower F1-score of **0.26** for this class.

The final Kaggle result of **0.47876** reflects the challenges in balancing performance across all classes. While the model demonstrated solid overall accuracy, further work could focus on improving predictions for the minority class, potentially through techniques such as oversampling, class weighting, or exploring different algorithms to better handle class imbalance.

Despite these challenges, the model shows promise, and with additional tuning, it could achieve even better results on unseen data.
